# ¿Se ha parado España? La demanda eléctrica durante el 11N

En este EDA comparo la demanda eléctrica real con la prevista por Red Eléctrica durante la huelga general del 11 de noviembre de 2026. No utilizo la diferencia como porcentaje de seguimiento: la electricidad mide actividad, no personas.

## 1. Preparación

Conservo la respuesta JSON original de REData. Cada serie contiene puntos de cinco minutos aunque la consulta utilice `time_trunc=hour`. Emparejo real y prevista por su fecha exacta.

In [1]:
from analisis_huelga_11n import cargar_series, resumir, guardar

bruto, filas = cargar_series()
print(f'Series comparables: {len(filas)}')
print(f'Actualización de REData: {bruto["data"]["attributes"]["last-update"]}')

Series comparables: 92
Actualización de REData: 2026-11-11T07:42:04.000+01:00


## 2. Diferencia frente a la previsión

Para cada instante calculo `(real - prevista) / prevista`. Una cifra negativa significa que la red registra menos demanda de la esperada. Para aproximar la energía acumulada integro cada diferencia durante cinco minutos.

In [2]:
resumen = resumir(filas)
guardar(filas, resumen)
for clave, valor in resumen.items():
    print(f'{clave}: {valor}')

ultima_actualizacion: 2026-11-11T07:35:00.000+01:00
observaciones: 92
real_mw: 28110.0
prevista_mw: 30607.0
diferencia_mw: -2497.0
diferencia_pct: -8.158
mayor_caida_pct: -8.812
momento_mayor_caida: 2026-11-11T07:00:00.000+01:00
deficit_acumulado_mwh: 7496.83
exceso_acumulado_mwh: 0.0
provisional: True


## 3. Lectura y límites

El resultado es provisional mientras siga abierta la jornada. La comparación correcta es con la previsión de REE, no con el día anterior. Aun así, el desvío puede incluir error de previsión, meteorología, autoconsumo y traslados de consumo por teletrabajo. No lo convierto en una tasa de seguimiento.

In [3]:
peor = min(filas, key=lambda fila: fila['diferencia_pct'])
print('Mayor caída observada en la copia analizada:')
print(f"{peor['datetime']}: {peor['diferencia_pct']:.2f}% ({peor['diferencia_mw']:.0f} MW)")
print('Conclusión: describo el desvío eléctrico; no estimo cuántas personas hicieron huelga.')

Mayor caída observada en la copia analizada:
2026-11-11T07:00:00.000+01:00: -8.81% (-2510 MW)
Conclusión: describo el desvío eléctrico; no estimo cuántas personas hicieron huelga.
